# Práctica 1 · Paso 2 — Delimitación de la cuenca, tabla morfométrica y ficha de la estación
### Hidrología (1CIA-502) · Ingeniería Ambiental · ESPAM MFL · Unidad I
**Docente:** Ing. Cristian Paliz Acosta, MSc.

**Qué vas a producir hoy**

1. El límite de **tu** cuenca, delimitado por ti desde el modelo digital de elevación (MDE ALOS PALSAR, 12,5 m).
2. La **tabla morfométrica** de la cuenca, con su interpretación.
3. La **ficha de la estación** pluviométrica que representa a tu cuenca, con su control de calidad.

**Regla del curso: mano antes que modelo.** Antes de ejecutar, escribe en la celda 1 los valores que
mediste a mano en el paso 1. Al final, el cuaderno compara tu medición con la del modelo y tú explicas la diferencia.

**Cómo trabajar:** `Archivo → Guardar una copia en Drive`. Ejecuta las celdas en orden con `Shift + Enter`.
Solo se edita la **celda 1**.

| Bloque | Qué hace | Producto |
|---|---|---|
| 0–2 | Librerías, configuración y descarga de capas desde GitHub | — |
| 3–5 | Acondicionamiento del MDE, dirección y acumulación de flujo | Mapas de flujo |
| 6 | Exutorio y delimitación | Límite de la cuenca |
| 7 | Red de drenaje, orden de Strahler y cauce principal | Mapa de drenaje |
| 8–9 | Parámetros morfométricos y curva hipsométrica | **Tabla morfométrica** |
| 10 | Comparación mano vs. modelo | Tabla de diferencias |
| 11 | Estación pluviométrica representativa | **Ficha de la estación** |
| 12 | Descarga de resultados | ZIP |

## 0 · Instalar librerías
Solo una vez por sesión (≈ 1 minuto).

In [ ]:
!pip install -q pysheds geopandas rasterio
import numpy as np
if not hasattr(np, "in1d"):      # compatibilidad de pysheds con NumPy 2.x
    np.in1d = np.isin
print("✅ Librerías listas")

## 1 · CONFIGURACIÓN ✍️ (única celda que editas)

- **CUENCA:** una de `Ayampe · Carrizal · Chone · Jama · Jipijapa · Portoviejo` (mayúscula inicial, sin tildes).
- **EXUTORIO:** coordenadas UTM 17S (m) de **tu** punto de salida, las del paso 1. Si lo dejas en `None`,
  el cuaderno usa el exutorio de la cuenca completa. Si eliges un punto aguas arriba, delimitas una subcuenca:
  así cada estudiante trabaja **su propia** cuenca.
- **AJUSTE_KM2:** tu exutorio se mueve al cauce más cercano que drene al menos esta área. Súbelo si tu punto
  cae sobre un afluente pequeño en lugar del río que quieres.
- **UMBRAL_KM2:** área mínima que debe drenar una celda para considerarla cauce. Cambia la red y la densidad de drenaje.
- **QUEMAR_CAUCES:** `True` graba la red de ríos de referencia en el MDE antes de calcular el flujo (sección 4).
  Ponlo en `False` para ver qué pasa sin esa corrección.
- **FACTOR_REMUESTREO:** 3 convierte la celda de 12,5 m en 37,5 m. Usa 2 si quieres más detalle; 4 si el cuaderno va lento.
- **MANO:** tus mediciones a mano del paso 1 (deja `None` lo que no hayas medido).

In [ ]:
ESTUDIANTE = "Apellidos Nombres"
CUENCA     = "Carrizal"

EXUTORIO   = None            # p. ej. (602100, 9902100) → subcuenca del Carrizal de ≈ 430 km² (Este, Norte; UTM 17S)
UMBRAL_KM2 = 1.0             # km² de área drenada para iniciar un cauce
AJUSTE_KM2 = 10.0            # el exutorio se lleva al cauce más cercano que drene al menos esta área
FACTOR_REMUESTREO = 3
QUEMAR_CAUCES = True         # graba el cauce principal oficial en el MDE (ver sección 4)
PROF_QUEMA = 20.0            # metros que se rebaja el cauce al quemarlo

# Mano antes que modelo: tus valores del paso 1
MANO = {"Área (km²)": None, "Perímetro (km)": None, "Longitud del cauce principal (km)": None,
        "Cota mínima (m)": None, "Cota máxima (m)": None}

ESTACION = None              # código INAMHI (p. ej. "M0164"); None = la más cercana al centroide

# Repositorio de datos del curso (no cambiar salvo indicación del docente)
DATOS_REPO = "crispaliz/datos-manabi"
DATOS_TAG  = "v1"

## 2 · Descargar las capas de tu cuenca desde GitHub
Se descargan del *release* del repositorio de datos `crispaliz/datos-manabi` a la sesión de Colab. No necesitas Drive.

In [ ]:
import os, zipfile, urllib.request, unicodedata
SLUG = unicodedata.normalize("NFKD", ESTUDIANTE).encode("ascii", "ignore").decode().replace(" ", "_")
url = f"https://github.com/{DATOS_REPO}/releases/download/{DATOS_TAG}/insumos_{CUENCA}.zip"
BASE = f"/content/insumos_{CUENCA}" if os.path.isdir("/content") else os.path.abspath(f"insumos_{CUENCA}")
if not os.path.exists(f"{BASE}/DEM_Cuencas_Manabi/DEM_{CUENCA}_limpio.tif"):
    print("Descargando", url)
    try:
        urllib.request.urlretrieve(url, "insumos.zip")
        with zipfile.ZipFile("insumos.zip") as z:
            z.extractall(BASE)
        os.remove("insumos.zip")
    except Exception as e:
        raise SystemExit(f"❌ Falló la descarga ({e}). Revisa el nombre de la cuenca en la celda 1.")
R = {"dem":        f"{BASE}/DEM_Cuencas_Manabi/DEM_{CUENCA}_limpio.tif",
     "oficial":    f"{BASE}/Cuencas_Manabi/cuenca_{CUENCA}.shp",
     "cauce":      f"{BASE}/Cauces_Manabi/cauce_principal_{CUENCA}.shp",
     "rios":       f"{BASE}/Rios_Manabi/rios_dem_{CUENCA}.shp",
     "estaciones": f"{BASE}/estaciones_precipitacion.csv"}
for k, v in R.items():
    print(("  ✅ " if os.path.exists(v) else "  ❌ FALTA "), k)
os.makedirs("salidas", exist_ok=True)

## 3 · El MDE de trabajo
Se remuestrea el MDE (promedio de celdas) y se marca como *sin dato* todo lo que está fuera del recorte.
Observa el relieve: ¿dónde están las partes altas que formarán la divisoria?

In [ ]:
import rasterio, geopandas as gpd, pandas as pd, matplotlib.pyplot as plt
from rasterio.enums import Resampling
from rasterio import features
from shapely.geometry import shape, LineString
import warnings; warnings.filterwarnings("ignore")

with rasterio.open(R["dem"]) as s:
    h, w = s.height // FACTOR_REMUESTREO, s.width // FACTOR_REMUESTREO
    Z = s.read(1, out_shape=(h, w), resampling=Resampling.average, masked=True).astype("float32").filled(np.nan)
    TR = s.transform * s.transform.scale(s.width / w, s.height / h)
    CRS = s.crs
Z[Z <= 0] = np.nan
CELDA = abs(TR.a); A_CELDA = CELDA**2 / 1e6
EXT = (TR.c, TR.c + w * TR.a, TR.f + h * TR.e, TR.f)
with rasterio.open("salidas/dem_trabajo.tif", "w", driver="GTiff", height=h, width=w, count=1,
                   dtype="float32", crs=CRS, transform=TR, nodata=-9999.0) as d:
    d.write(np.where(np.isnan(Z), -9999.0, Z), 1)

def sombreado(a, az=315, alt=45):
    gy, gx = np.gradient(a, CELDA)
    pend = np.pi/2 - np.arctan(np.hypot(gx, gy)); asp = np.arctan2(-gx, gy)
    azr, altr = np.radians(360 - az + 90), np.radians(alt)
    return np.sin(altr)*np.sin(pend) + np.cos(altr)*np.cos(pend)*np.cos(azr - asp)

oficial = gpd.read_file(R["oficial"]).to_crs(CRS)
fig, ax = plt.subplots(figsize=(9, 7))
ax.imshow(sombreado(Z), extent=EXT, cmap="gray")
im = ax.imshow(Z, extent=EXT, cmap="terrain", alpha=0.6)
plt.colorbar(im, ax=ax, shrink=0.7, label="Elevación (m s. n. m.)")
ax.set_title(f"MDE de trabajo · {CUENCA} · celda {CELDA:.1f} m", fontweight="bold")
ax.set_xlabel("Este (m)"); ax.set_ylabel("Norte (m)"); ax.ticklabel_format(style="plain")
plt.tight_layout(); plt.savefig("salidas/01_mde.png", dpi=150); plt.show()
print(f"Grilla {h} × {w} celdas · celda {CELDA:.1f} m · elevación {np.nanmin(Z):.0f}–{np.nanmax(Z):.0f} m")

## 4 · Acondicionamiento hidrológico del MDE
Un MDE tiene **depresiones** (celdas más bajas que todas sus vecinas) y **zonas planas**: el agua quedaría
atrapada y el flujo se interrumpiría. Además, en gargantas angostas con vegetación el radar puede registrar
la copa de los árboles o las laderas en lugar del cauce, y aparece un **dique falso** que cierra el valle.

El acondicionamiento tiene cuatro pasos:
1. **Quemado de cauces** (*stream burning*): se graba la red de ríos de referencia en el MDE, bajando esas celdas,
   para que el flujo siga los cauces conocidos y atraviese los diques falsos.
2. Rellenar hoyos aislados.
3. Rellenar depresiones.
4. Dar una pendiente mínima a las zonas planas.

Las cotas que se usan después en la tabla morfométrica salen siempre del MDE **original**, no del quemado.

In [ ]:
from pysheds.grid import Grid
rios_ref = gpd.read_file(R["cauce"]).to_crs(CRS)   # solo el cauce principal oficial
QUEMA = features.rasterize([(g, 1) for g in rios_ref.geometry], out_shape=Z.shape, transform=TR,
                           fill=0, all_touched=True).astype(bool) & np.isfinite(Z)
ZQ = np.where(QUEMA, Z - PROF_QUEMA, Z) if QUEMAR_CAUCES else Z.copy()
with rasterio.open("salidas/dem_acondicionado_entrada.tif", "w", driver="GTiff", height=h, width=w, count=1,
                   dtype="float32", crs=CRS, transform=TR, nodata=-9999.0) as d:
    d.write(np.where(np.isnan(ZQ), -9999.0, ZQ), 1)
print(f"Quemado de cauces: {'sí' if QUEMAR_CAUCES else 'no'} · celdas de cauce de referencia: {QUEMA.sum():,}")

grid = Grid.from_raster("salidas/dem_acondicionado_entrada.tif")
dem  = grid.read_raster("salidas/dem_acondicionado_entrada.tif")
dem_sin_hoyos = grid.fill_pits(dem)
dem_relleno   = grid.fill_depressions(dem_sin_hoyos)
dem_hidro     = grid.resolve_flats(dem_relleno)
relleno = np.asarray(dem_relleno) - np.asarray(dem)
relleno[np.isnan(Z) | QUEMA] = np.nan
print(f"Celdas corregidas por relleno: {(relleno > 0.01).sum():,} "
      f"({100*(relleno > 0.01).sum()/np.isfinite(Z).sum():.1f} % del MDE) · relleno máximo {np.nanmax(relleno):.1f} m")

## 5 · Dirección y acumulación de flujo (D8)
Cada celda envía su agua a **una** de sus 8 vecinas: la de mayor pendiente descendente. La **acumulación**
cuenta cuántas celdas drenan a través de cada celda: multiplicada por el área de la celda, es el área drenada.

In [ ]:
DIRMAP = (64, 128, 1, 2, 4, 8, 16, 32)          # N, NE, E, SE, S, SO, O, NO
fdir = grid.flowdir(dem_hidro, dirmap=DIRMAP)
acc  = grid.accumulation(fdir, dirmap=DIRMAP)
ACC = np.asarray(acc, dtype="float64"); ACC[np.isnan(Z)] = np.nan
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(np.log10(ACC * A_CELDA), extent=EXT, cmap="Blues", vmin=-2)
plt.colorbar(im, ax=ax, shrink=0.7, label="log₁₀ del área drenada (km²)")
oficial.boundary.plot(ax=ax, color="k", lw=0.8)
ax.set_title(f"Acumulación de flujo · {CUENCA}", fontweight="bold")
ax.set_xlabel("Este (m)"); ax.set_ylabel("Norte (m)"); ax.ticklabel_format(style="plain")
plt.tight_layout(); plt.savefig("salidas/02_acumulacion.png", dpi=150); plt.show()

## 6 · Exutorio y delimitación
El exutorio se **ajusta** a la celda de cauce más cercana (si cae fuera del cauce, la cuenca delimitada sería
de unas pocas celdas). Luego se marcan todas las celdas cuyo flujo pasa por él: esa es la cuenca.
El cuaderno compara tu delimitación con el límite oficial del curso.

In [ ]:
UMBRAL_CELDAS = UMBRAL_KM2 / A_CELDA
mask_of = features.rasterize([(g, 1) for g in oficial.geometry], out_shape=Z.shape, transform=TR, fill=0).astype(bool)
if EXUTORIO is None:
    r0, c0 = np.unravel_index(np.nanargmax(np.where(mask_of, ACC, -1)), ACC.shape)
    x0, y0 = TR * (c0 + 0.5, r0 + 0.5)
    print("Exutorio automático: celda de mayor acumulación dentro del límite oficial")
else:
    x0, y0 = EXUTORIO
umbral_ajuste = max(UMBRAL_KM2, AJUSTE_KM2 if EXUTORIO is not None else UMBRAL_KM2) / A_CELDA
XO, YO = map(float, grid.snap_to_mask(acc > umbral_ajuste, np.array([[x0, y0]]))[0])
print(f"Exutorio ajustado al cauce: E {XO:,.0f} m · N {YO:,.0f} m (se movió {np.hypot(XO-x0, YO-y0):.0f} m)")

catch = grid.catchment(x=XO, y=YO, fdir=fdir, dirmap=DIRMAP, xytype="coordinate")
CM = np.asarray(catch).astype(bool)
partes = [shape(g) for g, v in features.shapes(CM.astype("uint8"), mask=CM, transform=TR) if v == 1]
poli_raster = max(partes, key=lambda g: g.area)
# El borde ráster es una escalera de celdas: se suaviza con una tolerancia de una celda
cuenca = gpd.GeoDataFrame({"cuenca": [CUENCA], "estudiante": [ESTUDIANTE]},
                          geometry=[poli_raster.simplify(CELDA)], crs=CRS)
P_escalera = poli_raster.length / 1000

lim_of = oficial.union_all()
A_of  = lim_of.area / 1e6
inter = cuenca.geometry.iloc[0].intersection(lim_of).area / 1e6
union = cuenca.geometry.iloc[0].union(lim_of).area / 1e6
if EXUTORIO is None:
    print(f"Área delimitada: {cuenca.area.iloc[0]/1e6:,.1f} km² · área oficial: {A_of:,.1f} km² · "
          f"coincidencia (IoU): {inter/union:.2f}")
else:
    print(f"Subcuenca delimitada: {cuenca.area.iloc[0]/1e6:,.1f} km² · "
          f"{100*inter/(cuenca.area.iloc[0]/1e6):.0f} % de ella queda dentro del límite oficial de {CUENCA}")
print(f"Perímetro: {cuenca.length.iloc[0]/1000:,.1f} km suavizado · {P_escalera:,.1f} km con el borde en escalera · "
      f"{lim_of.length/1000:,.1f} km el límite oficial")

fig, ax = plt.subplots(figsize=(9, 7))
ax.imshow(sombreado(Z), extent=EXT, cmap="gray")
ax.imshow(np.where(CM, 1, np.nan), extent=EXT, cmap="autumn", alpha=0.35)
oficial.boundary.plot(ax=ax, color="k", lw=1, ls="--", label="Límite oficial")
cuenca.boundary.plot(ax=ax, color="crimson", lw=1.6, label="Tu delimitación")
ax.plot(XO, YO, marker="*", color="crimson", ms=16, mec="k", label="Exutorio")
ax.legend(loc="lower left"); ax.set_title(f"Delimitación desde el MDE · {CUENCA}", fontweight="bold")
ax.set_xlabel("Este (m)"); ax.set_ylabel("Norte (m)"); ax.ticklabel_format(style="plain")
plt.tight_layout(); plt.savefig("salidas/03_delimitacion.png", dpi=150); plt.show()

## 7 · Red de drenaje, orden de Strahler y cauce principal
- **Red:** celdas cuya área drenada supera `UMBRAL_KM2`.
- **Orden de Strahler:** dos cauces de orden *n* que se unen forman uno de orden *n + 1*.
- **Cauce principal:** el recorrido de flujo más largo, desde la divisoria hasta el exutorio.

In [ ]:
RED = (ACC > UMBRAL_CELDAS) & CM
orden = np.asarray(grid.stream_order(fdir, acc > UMBRAL_CELDAS, dirmap=DIRMAP), dtype=float)
orden[~RED] = np.nan
FD = np.asarray(fdir)
MOV = {64: (-1, 0), 128: (-1, 1), 1: (0, 1), 2: (1, 1), 4: (1, 0), 8: (1, -1), 16: (0, -1), 32: (-1, -1)}
PASO = np.zeros(FD.shape)
for k, (dr, dc) in MOV.items():
    PASO[FD == k] = CELDA * (np.sqrt(2) if dr and dc else 1.0)

# Recorrido más largo: distancia ponderada (en metros) hasta el exutorio
from pysheds.sview import Raster
pesos = Raster(PASO, viewfinder=fdir.viewfinder)
dist = np.asarray(grid.distance_to_outlet(x=XO, y=YO, fdir=fdir, weights=pesos, dirmap=DIRMAP,
                                          xytype="coordinate"), dtype=float)
dist[~CM] = np.nan
r, c = np.unravel_index(np.nanargmax(dist), dist.shape)
camino = [(r, c)]
while dist[r, c] > 0 and len(camino) < Z.size:
    dr, dc = MOV[int(FD[r, c])]; r, c = r + dr, c + dc; camino.append((r, c))
L_m = sum(PASO[p] for p in camino[:-1])
z_cab, z_sal = float(Z[camino[0]]), float(Z[camino[-1]])   # cotas del MDE original
cauce_ppal = gpd.GeoDataFrame({"tipo": ["cauce principal"]},
    geometry=[LineString([TR * (cc + 0.5, rr + 0.5) for rr, cc in camino])], crs=CRS)

# Red en formato vectorial, con el orden de Strahler de cada tramo
rn = grid.extract_river_network(fdir, acc > UMBRAL_CELDAS, dirmap=DIRMAP)
red = gpd.GeoDataFrame.from_features(rn["features"], crs=CRS)
red = red[red.intersects(cuenca.geometry.iloc[0])].copy()
def _orden(geom):
    x, y = geom.interpolate(0.5, normalized=True).coords[0]
    cc, rr = ~TR * (x, y); rr, cc = int(rr), int(cc)
    v = orden[rr, cc] if 0 <= rr < orden.shape[0] and 0 <= cc < orden.shape[1] else np.nan
    return int(v) if np.isfinite(v) else 1
red["orden"] = red.geometry.apply(_orden)
red = red[["orden", "geometry"]]
NMAX = int(np.nanmax(orden))
cmap_o = plt.get_cmap("viridis_r", NMAX)

fig, ax = plt.subplots(figsize=(9, 7))
ax.imshow(sombreado(np.where(CM, Z, np.nan)), extent=EXT, cmap="gray", alpha=0.5)
for n in range(1, NMAX + 1):
    red[red.orden == n].plot(ax=ax, color=cmap_o(n - 1), lw=0.3 + 0.45 * n, label=f"Orden {n}")
cauce_ppal.plot(ax=ax, color="crimson", lw=1.4, ls="--", label="Cauce principal")
cuenca.boundary.plot(ax=ax, color="k", lw=1)
ax.legend(loc="lower left", fontsize=8)
ax.set_title(f"Red de drenaje (umbral {UMBRAL_KM2:g} km²) · {CUENCA}".replace(".", ","), fontweight="bold")
b = cuenca.total_bounds; ax.set_xlim(b[0]-2000, b[2]+2000); ax.set_ylim(b[1]-2000, b[3]+2000)
ax.set_xlabel("Este (m)"); ax.set_ylabel("Norte (m)"); ax.ticklabel_format(style="plain")
plt.tight_layout(); plt.savefig("salidas/04_red_drenaje.png", dpi=150); plt.show()
print(f"Cauce principal: {L_m/1000:.2f} km · de {z_cab:.0f} m a {z_sal:.0f} m · orden máximo {int(np.nanmax(orden))}")

## 8 · Tabla morfométrica
| Parámetro | Fórmula |
|---|---|
| Coeficiente de compacidad | $K_c = 0{,}28\,P/\sqrt{A}$ |
| Factor de forma | $F_f = A/L^2$ |
| Relación de elongación (Schumm) | $R_e = 1{,}128\,\sqrt{A}/L$ |
| Densidad de drenaje | $D_d = L_t/A$ |
| Pendiente del cauce principal | $S = (z_{cab} - z_{sal})/L$ |
| Tiempo de concentración (Kirpich) | $t_c\,[\text{min}] = 0{,}0195\,L^{0,77}\,S^{-0,385}$, con $L$ en m |
| Integral hipsométrica | $IH = (\bar z - z_{min})/(z_{max} - z_{min})$ |

In [ ]:
A  = cuenca.area.iloc[0] / 1e6
P  = cuenca.length.iloc[0] / 1000
L  = L_m / 1000
Lt = PASO[RED].sum() / 1000
zc = Z[CM]; zc = zc[np.isfinite(zc)]
zmin, zmax, zmed = zc.min(), zc.max(), zc.mean()
gy, gx = np.gradient(Z, CELDA)
pend_media = np.nanmean(np.hypot(gx, gy)[CM]) * 100
Kc = 0.28 * P / np.sqrt(A); Ff = A / L**2; Re = 1.128 * np.sqrt(A) / L; Dd = Lt / A
S  = (z_cab - z_sal) / L_m
tc = 0.0195 * L_m**0.77 * S**-0.385
IH = (zmed - zmin) / (zmax - zmin)

def forma(k):
    return ("redonda a oval-redonda" if k < 1.25 else
            "oval-redonda a oval-oblonga" if k < 1.5 else "oval-oblonga a rectangular-oblonga")
def etapa(ih):
    return "joven (en desequilibrio)" if ih > 0.6 else ("madura (en equilibrio)" if ih >= 0.35 else "senil (en vejez)")

filas = [   # (parámetro, símbolo, valor, decimales, unidad, lectura)
 ("Área", "A", A, 1, "km²", ""),
 ("Perímetro", "P", P, 1, "km", "borde suavizado a una celda"),
 ("Longitud del cauce principal", "L", L, 2, "km", "recorrido de flujo más largo"),
 ("Longitud total de cauces", "Lt", Lt, 1, "km", f"umbral de {UMBRAL_KM2:g} km²".replace(".", ",")),
 ("Coeficiente de compacidad", "Kc", Kc, 2, "—", forma(Kc)),
 ("Factor de forma", "Ff", Ff, 3, "—", "más bajo, más alargada"),
 ("Relación de elongación", "Re", Re, 2, "—", "cercana a 1: casi circular"),
 ("Densidad de drenaje", "Dd", Dd, 2, "km/km²", "depende del umbral"),
 ("Orden de Strahler máximo", "N", np.nanmax(orden), 0, "—", ""),
 ("Cota mínima", "zmín", zmin, 0, "m", ""),
 ("Cota máxima", "zmáx", zmax, 0, "m", ""),
 ("Cota media", "z̄", zmed, 0, "m", ""),
 ("Relieve", "ΔH", zmax - zmin, 0, "m", ""),
 ("Pendiente media de la cuenca", "Sc", pend_media, 1, "%", ""),
 ("Pendiente del cauce principal", "S", S, 4, "m/m", f"de {z_cab:.0f} a {z_sal:.0f} m"),
 ("Tiempo de concentración (Kirpich)", "tc", tc / 60, 1, "h", f"{tc:.0f} min"),
 ("Integral hipsométrica", "IH", IH, 2, "—", etapa(IH)),
]
MORF = pd.DataFrame([(p_, s_, f"{v:,.{d}f}".replace(",", " ").replace(".", ","), u, l)
                     for p_, s_, v, d, u, l in filas],
                    columns=["Parámetro", "Símbolo", "Valor", "Unidad", "Lectura"])
MORF.to_csv(f"salidas/tabla_morfometrica_{CUENCA}_{SLUG}.csv", index=False, encoding="utf-8-sig")
MORF

## 9 · Curva hipsométrica
Distribución del área de la cuenca según la altitud.

In [ ]:
niveles = np.linspace(zmin, zmax, 50)
pct = np.array([(zc >= n).mean() * 100 for n in niveles])
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(pct, niveles, color="#0B4F6C", lw=2); ax.fill_betweenx(niveles, pct, color="#0E7C86", alpha=0.15)
ax.axhline(zmed, color="crimson", ls="--", lw=1, label=f"Cota media {zmed:.0f} m")
ax.set_xlabel("% del área por encima de la cota"); ax.set_ylabel("Cota (m s. n. m.)")
ax.set_title(f"Curva hipsométrica · {CUENCA} · IH = {IH:.2f}", fontweight="bold"); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.savefig("salidas/05_hipsometrica.png", dpi=150); plt.show()

## 10 · Mano antes que modelo
Compara lo que mediste a mano en el paso 1 con lo que calculó el modelo. En tu bitácora, explica **por qué**
difieren (escala del mapa, trazado de la divisoria, tamaño de la celda, zonas planas, umbral de la red…).

In [ ]:
modelo = {"Área (km²)": A, "Perímetro (km)": P, "Longitud del cauce principal (km)": L,
          "Cota mínima (m)": zmin, "Cota máxima (m)": zmax}
comp = pd.DataFrame([(k, MANO.get(k), round(v, 2),
                      None if MANO.get(k) in (None, 0) else round(100 * (v - MANO[k]) / MANO[k], 1))
                     for k, v in modelo.items()], columns=["Parámetro", "A mano", "Modelo", "Diferencia (%)"])
if all(v is None for v in MANO.values()):
    print("⚠️  No escribiste tus valores a mano en la celda 1. Complétalos y vuelve a ejecutar esta celda.")
comp

## 11 · Ficha de la estación pluviométrica
Se usan las estaciones del INAMHI del archivo del curso. El cuaderno elige la más cercana al centroide de tu
cuenca (o la que indiques en `ESTACION`) y revisa la **calidad del dato** antes de usarla:
altitud registrada frente al MDE, longitud de la serie, series alternativas de la misma estación y
coherencia con las estaciones vecinas.

In [ ]:
# Datos crudos: catálogo nacional del INAMHI publicado en el repositorio del curso
URL_EST = f"https://raw.githubusercontent.com/{DATOS_REPO}/main/estaciones_precipitacion.csv"
est = pd.read_csv(URL_EST)
# Serie depurada del curso (filtro de valores atípicos espaciales), incluida en tus insumos
dep = pd.read_csv(R["estaciones"])
gest = gpd.GeoDataFrame(est, geometry=gpd.points_from_xy(est.X, est.Y), crs=4326).to_crs(CRS)
gest["depurada"] = gest.Code.isin(dep.codigo)
cen = cuenca.geometry.iloc[0].centroid
gest["dist_km"] = gest.distance(cen) / 1000
gest["dentro"] = gest.within(cuenca.geometry.iloc[0])
with rasterio.open(R["dem"]) as s:
    gest["z_mde"] = [v[0] if v[0] > 0 else np.nan for v in s.sample([(p.x, p.y) for p in gest.geometry])]
cerca = gest.sort_values("dist_km").head(8)
print("Estaciones más cercanas al centroide de la cuenca:")
display(cerca[["Code", "Name", "Z", "z_mde", "P_anual", "meses_secos", "serie_alterna", "depurada",
               "dist_km", "dentro"]].round(1))

e = gest[gest.Code == ESTACION].iloc[0] if ESTACION else cerca.iloc[0]
mismas = gest[(gest.X == e.X) & (gest.Y == e.Y) & (gest.Code != e.Code)]
vecinas = gest[(gest.dist_km < 25) & (gest.serie_alterna == 0) & (gest.Code != e.Code)]

alertas = []
if e.Z == 0:
    alertas.append(f"Altitud registrada 0 m: dato faltante. El MDE indica ≈ {e.z_mde:.0f} m.")
elif np.isfinite(e.z_mde) and abs(e.Z - e.z_mde) > 50:
    alertas.append(f"La altitud registrada ({e.Z:.0f} m) difiere {abs(e.Z - e.z_mde):.0f} m del MDE: revisar coordenadas.")
if e.serie_alterna == 1:
    alertas.append("Es una serie alternativa (código con sufijo): verificar cuál serie es la oficial.")
for _, m in mismas.iterrows():
    alertas.append(f"Comparte coordenadas con {m.Code} ({m.Name}): P anual {m.P_anual:.0f} mm "
                   f"frente a {e.P_anual:.0f} mm. Diferencia de {abs(m.P_anual - e.P_anual):.0f} mm.")
if len(vecinas):
    med = vecinas.P_anual.median()
    if abs(e.P_anual - med) / med > 0.4:
        alertas.append(f"La P anual difiere más de 40 % de la mediana de las vecinas a < 25 km ({med:.0f} mm).")
if not e.depurada:
    alertas.append("No pasó el filtro de calidad del curso: no figura en la serie depurada.")
if not e.dentro:
    alertas.append("La estación está fuera de la cuenca: su representatividad es limitada.")

meses = ["E", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"]
pm = np.array([e[f"P_m{i:02d}"] for i in range(1, 13)], dtype=float)
fig = plt.figure(figsize=(11, 4.8))
ax1 = fig.add_axes([0.05, 0.12, 0.42, 0.75])
ax1.bar(range(12), pm, color=["#0E7C86" if v >= 60 else "#E07A1F" for v in pm])
ax1.axhline(60, color="k", lw=0.8, ls=":"); ax1.text(-0.4, 63, "60 mm", ha="left", fontsize=8)
ax1.set_xticks(range(12)); ax1.set_xticklabels(meses); ax1.set_ylabel("Precipitación media (mm)")
ax1.set_title(f"Régimen mensual · {e.Code} {e.Name}", fontweight="bold")
ax2 = fig.add_axes([0.52, 0.05, 0.46, 0.85]); ax2.axis("off")
txt = [f"FICHA DE LA ESTACIÓN — {e.Code} {e.Name}",
       f"Cuenca: {CUENCA} · Estudiante: {ESTUDIANTE}", "",
       f"Coordenadas: {e.Y:.4f}°, {e.X:.4f}°",
       f"Altitud registrada: {e.Z:.0f} m · según MDE: {e.z_mde:.0f} m",
       f"Ubicación: {'dentro' if e.dentro else 'fuera'} de la cuenca · {e.dist_km:.1f} km del centroide",
       f"Años de registro: {e.n_anios}",
       f"P anual media: {e.P_anual:.0f} mm (mín. {e.P_min_anual:.0f} · máx. {e.P_max_anual:.0f})",
       f"Meses secos (< 60 mm): {e.meses_secos}",
       f"Mes más lluvioso: {meses[int(pm.argmax())]} ({pm.max():.0f} mm) · más seco: {meses[int(pm.argmin())]} ({pm.min():.0f} mm)",
       f"En la serie depurada del curso: {'sí' if e.depurada else 'no'}", "", "CONTROL DE CALIDAD"] + ([f"• {a}" for a in alertas] or ["• Sin alertas automáticas. Revisa igual la serie."])
import textwrap
y = 1.0
for i, t in enumerate(txt):
    for lin in textwrap.wrap(t, 70) or [""]:
        ax2.text(0, y, lin, fontsize=9.5, va="top", fontweight="bold" if t.startswith(("FICHA", "CONTROL")) else "normal",
                 color="crimson" if t.startswith("•") and alertas else "black"); y -= 0.062
plt.savefig(f"salidas/06_ficha_estacion_{e.Code}.png", dpi=150, bbox_inches="tight"); plt.show()
pd.DataFrame({"Campo": [t.split(":")[0] for t in txt if ":" in t],
              "Valor": [t.split(":", 1)[1].strip() for t in txt if ":" in t]}).to_csv(
    f"salidas/ficha_estacion_{e.Code}.csv", index=False, encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(7, 6))
cuenca.boundary.plot(ax=ax, color="k", lw=1.2)
gest[gest.dist_km < 40].plot(ax=ax, color="steelblue", markersize=18)
gpd.GeoSeries([e.geometry], crs=CRS).plot(ax=ax, color="crimson", markersize=90, marker="^")
for _, s_ in gest[gest.dist_km < 40].iterrows():
    ax.annotate(s_.Code, (s_.geometry.x, s_.geometry.y), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.plot(cen.x, cen.y, "k+", ms=12)
ax.set_title(f"Estaciones alrededor de la cuenca {CUENCA}", fontweight="bold")
ax.set_xlabel("Este (m)"); ax.set_ylabel("Norte (m)"); ax.ticklabel_format(style="plain")
plt.tight_layout(); plt.savefig("salidas/07_estaciones.png", dpi=150); plt.show()

## 12 · Descargar tus resultados
Límite y cauce principal en GeoPackage (se abren en QGIS), tabla morfométrica, ficha y figuras.

In [ ]:
import shutil
cuenca.to_file(f"salidas/cuenca_{CUENCA}_{SLUG}.gpkg", layer="cuenca", driver="GPKG")
cauce_ppal.to_file(f"salidas/cuenca_{CUENCA}_{SLUG}.gpkg", layer="cauce_principal", driver="GPKG")
red.to_file(f"salidas/cuenca_{CUENCA}_{SLUG}.gpkg", layer="red_drenaje", driver="GPKG")
paquete = shutil.make_archive(f"Practica1_Paso2_{CUENCA}_{SLUG}", "zip", "salidas")
print("✅ Paquete:", paquete)
try:
    from google.colab import files; files.download(paquete)
except Exception:
    print("Descárgalo desde el panel de archivos de Colab.")

---
## Para tu bitácora y la ficha (entrega)
1. Con tu **Kc**, tu **Ff** y tu **tc**, describe en tres líneas cómo respondería tu cuenca a un aguacero intenso de El Niño.
2. Cambia `UMBRAL_KM2` a 0,5 y a 2 y vuelve a ejecutar desde la celda 6. ¿Qué parámetros cambian y cuáles no? ¿Por qué?
3. Ejecuta con `QUEMAR_CAUCES = False` y compara: ¿cambia tu cuenca? ¿Por qué? ¿Coincide tu delimitación con el límite oficial? Señala en el mapa dónde no coinciden y explica la causa probable.
4. ¿La estación de tu ficha representa bien a tu cuenca? Justifica con su ubicación, su altitud y su control de calidad.